In [ ]:
import pandas as pd
import os
from google.cloud import bigquery


In [17]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "service_client.json"
client = bigquery.Client()

QUERY = """
WITH 
block_stats AS (
  SELECT
    AVG(SAFE_DIVIDE(gas_used, gas_limit)) * 100 AS avg_block_utilization_pct
  FROM `bigquery-public-data.goog_blockchain_ethereum_mainnet_us.blocks`
  WHERE DATE(block_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
),
tx_stats AS (
  SELECT
    COUNT(*) AS total_transactions,
    AVG((CAST(gas AS BIGNUMERIC) * gas_price) / POWER(10,18) * 1600) AS avg_gas_cost_usd,
    SUM(value) / POWER(10,18) AS total_eth_transferred
  FROM `bigquery-public-data.goog_blockchain_ethereum_mainnet_us.transactions`
  WHERE DATE(block_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
),
active_address_stats AS (
  SELECT
    COUNT(DISTINCT addr) AS active_unique_addresses
  FROM `bigquery-public-data.goog_blockchain_ethereum_mainnet_us.transactions`,
  UNNEST([from_address, to_address]) AS addr
  WHERE DATE(block_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
),
receipt_stats AS (
  SELECT
    ROUND(SAFE_DIVIDE(COUNTIF(status = 0), COUNT(*)) * 100, 3) AS failed_tx_rate_pct,
    COUNT(DISTINCT contract_address) AS new_contracts_deployed
  FROM `bigquery-public-data.goog_blockchain_ethereum_mainnet_us.receipts`
  WHERE DATE(block_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
),
token_stats AS (
  SELECT
    COUNT(*) AS token_transfer_count
  FROM `bigquery-public-data.goog_blockchain_ethereum_mainnet_us.token_transfers`
  WHERE DATE(block_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
)
SELECT
  DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY) AS date,
  tx_stats.total_transactions,
  tx_stats.avg_gas_cost_usd,
  tx_stats.total_eth_transferred,
  active_address_stats.active_unique_addresses,
  block_stats.avg_block_utilization_pct,
  receipt_stats.failed_tx_rate_pct,
  receipt_stats.new_contracts_deployed,
  token_stats.token_transfer_count
FROM tx_stats, active_address_stats, block_stats, receipt_stats, token_stats
"""

In [18]:




def fetch_daily_metrics():
    return client.query(QUERY).to_dataframe()

def save_locally(df_today, path="data/eth_daily_metrics.csv"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df_today["date"] = pd.to_datetime(df_today["date"]).dt.date

    if os.path.exists(path):
        df_existing = pd.read_csv(path)
        df_existing["date"] = pd.to_datetime(df_existing["date"]).dt.date
        df_combined = pd.concat([df_existing, df_today]).drop_duplicates(subset="date", keep="last")
    else:
        df_combined = df_today

    df_combined.to_csv(path, index=False)
    return df_combined

def upload_to_bigquery(df_combined, table_id):
    project_id, dataset_id, table_name = table_id.split(".")

    dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
    dataset_ref.location = "US"
    client.create_dataset(dataset_ref, exists_ok=True)

    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    job = client.load_table_from_dataframe(df_combined, table_id, job_config=job_config)
    job.result()

if __name__ == "__main__":
    df_today = fetch_daily_metrics()
    df_combined = save_locally(df_today)
    upload_to_bigquery(df_combined, "project-new-499509.data.eth_daily_metrics")
    print(f"Fetched {len(df_today)} new row(s). Total history: {len(df_combined)} rows.")

Fetched 1 new row(s). Total history: 1 rows.
